# Requetes metier - Exploitation du graphe Neo4j
## Projet SAE : Migration SQL vers Neo4j - Crimes et Delits (2012-2022)

Ce notebook liste les requetes Cypher a executer dans **Neo4j Browser** (`http://localhost:7474`).

Pour chaque requete :
1. Copier la requete Cypher dans Neo4j Browser
2. Executer et observer le resultat (graphe de noeuds)
3. Coller la capture d'ecran dans l'emplacement prevu

### Connexion Neo4j Browser
- URL : `http://localhost:7474`
- User : `neo4j`
- Password : `admin1234`

---
## 1. Vue d'ensemble du graphe - Comptage des noeuds et relations

**Contexte metier** : Verifier l'integrite des donnees apres migration.

```cypher
MATCH (n)
RETURN labels(n)[0] AS label, COUNT(n) AS nombre
ORDER BY nombre DESC
```

```cypher
MATCH ()-[r]->()
RETURN type(r) AS relation, COUNT(r) AS nombre
ORDER BY nombre DESC
```

**Capture d'ecran :**

*( coller ici la capture de Neo4j Browser )*

---
## 2. Visualisation du modele complet (exemple d'un sous-graphe)

**Contexte metier** : Montrer tous les types de noeuds et relations du modele — Region, Departement, Service, Perimetre, Enregistrement, Infraction.

```cypher
MATCH (s:Service)-[:SE_TROUVE]->(d:Departement)-[:APPARTIENT_A]->(r:Region),
      (s)-[:APPARTIENT]->(p:Perimetre),
      (s)-[:ENREGISTRE]->(e:Enregistrement)-[:CONCERNE]->(i:Infraction)
RETURN s, d, r, p, e, i
LIMIT 5
```

**Capture d'ecran :**

*( coller ici la capture de Neo4j Browser )*

---
## 3. Top 10 des infractions les plus frequentes

**Contexte metier** : Identifier les infractions les plus courantes sur l'ensemble du territoire pour orienter les politiques de prevention.

```cypher
MATCH (e:Enregistrement)-[:CONCERNE]->(i:Infraction)
RETURN i.libelle AS infraction, SUM(e.nb_faits) AS total_faits
ORDER BY total_faits DESC
LIMIT 10
```

**Capture d'ecran :**

*( coller ici la capture de Neo4j Browser )*

---
## 4. Crimes les plus frequents par departement (Top 3)

**Contexte metier** : Identifier pour chaque departement les infractions dominantes afin d'adapter les moyens locaux.

```cypher
MATCH (s:Service)-[:SE_TROUVE]->(d:Departement),
      (s)-[:ENREGISTRE]->(e:Enregistrement)-[:CONCERNE]->(i:Infraction)
WITH d.nom_dept AS departement, i.libelle AS infraction, SUM(e.nb_faits) AS total
ORDER BY departement, total DESC
WITH departement, COLLECT({infraction: infraction, total: total})[0..3] AS top3
UNWIND top3 AS t
RETURN departement, t.infraction AS infraction, t.total AS total_faits
```

**Capture d'ecran :**

*( coller ici la capture de Neo4j Browser )*

---
## 5. Structure hierarchique : Regions et leurs Departements

**Contexte metier** : Visualiser la relation `APPARTIENT_A` entre departements et regions sous forme de graphe de noeuds.

```cypher
MATCH (d:Departement)-[:APPARTIENT_A]->(r:Region)
RETURN d, r
```

**Capture d'ecran :**

*( coller ici la capture de Neo4j Browser )*

---
## 6. Connexions Services - Departements (zoom sur une region)

**Contexte metier** : Analyser le maillage territorial — quels services couvrent quels departements dans une region donnee.

```cypher
MATCH (s:Service)-[:SE_TROUVE]->(d:Departement)-[:APPARTIENT_A]->(r:Region)
WHERE r.nom_region = 'Ile-de-France'
RETURN s, d, r
LIMIT 50
```

**Capture d'ecran :**

*( coller ici la capture de Neo4j Browser )*

---
## 7. Repartition Police / Gendarmerie pour un departement

**Contexte metier** : Visualiser les services rattaches a un departement et leur perimetre (Police nationale ou Gendarmerie nationale).

```cypher
MATCH (s:Service)-[:SE_TROUVE]->(d:Departement),
      (s)-[:APPARTIENT]->(p:Perimetre)
WHERE d.nom_dept = 'Paris'
RETURN s, d, p
```

**Capture d'ecran :**

*( coller ici la capture de Neo4j Browser )*

---
## 8. Adjacences geographiques d'un departement

**Contexte metier** : Exploiter la relation `EST_ADJACENT` — avantage cle du modele graphe — pour visualiser les departements voisins.

```cypher
MATCH (d:Departement {code_dept: '75'})-[:EST_ADJACENT]-(voisin:Departement)
RETURN d, voisin
```

**Capture d'ecran :**

*( coller ici la capture de Neo4j Browser )*

---
## 9. Graphe complet des adjacences entre departements

**Contexte metier** : Visualiser l'integralite du reseau d'adjacences geographiques — une vue uniquement possible grace au modele graphe.

```cypher
MATCH (a:Departement)-[:EST_ADJACENT]->(b:Departement)
RETURN a, b
```

**Capture d'ecran :**

*( coller ici la capture de Neo4j Browser )*

---
## 10. Plus court chemin entre deux departements (shortestPath)

**Contexte metier** : Demontrer la puissance du modele graphe — trouver le plus court chemin geographique entre deux departements via les adjacences. Impossible en SQL sans requetes recursives complexes.

```cypher
MATCH (a:Departement {code_dept: '75'}),
      (b:Departement {code_dept: '13'}),
      path = shortestPath((a)-[:EST_ADJACENT*]-(b))
RETURN path
```

**Capture d'ecran :**

*( coller ici la capture de Neo4j Browser )*

---
## 11. Enregistrements d'un service avec les infractions concernees

**Contexte metier** : Explorer le detail des enregistrements d'un service specifique — quelles infractions il a enregistrees, combien de faits.

```cypher
MATCH (s:Service)-[:ENREGISTRE]->(e:Enregistrement)-[:CONCERNE]->(i:Infraction)
WHERE s.code_service = 'SVC-00001'
RETURN s, e, i
LIMIT 20
```

**Capture d'ecran :**

*( coller ici la capture de Neo4j Browser )*

---
## Conclusion

Ces requetes demontrent la puissance du modele graphe pour :
- **Traversees multi-niveaux** : Service -> Departement -> Region en une seule requete
- **Analyse de voisinage** : exploiter les adjacences geographiques (`EST_ADJACENT`)
- **Plus courts chemins** : `shortestPath` natif, impossible en SQL standard
- **Visualisation naturelle** : les resultats s'affichent directement sous forme de noeuds et relations dans Neo4j Browser

Le modele relationnel necessiterait des jointures multiples et des CTE recursives pour obtenir les memes resultats.